In [ ]:
# Cell 1: Install dependencies
!apt-get update -q && apt-get install -q zstd curl -y
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q pinggy fastapi uvicorn ollama pyngrok

In [ ]:
# Cell 2: Start Ollama and pull model
import os
import subprocess
import time
import requests

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'

print("--> Cleaning up existing processes...")
subprocess.run(["pkill", "ollama"], capture_output=True)
time.sleep(2)
os.environ['OLLAMA_NUM_PARALLEL'] = '10'
os.environ['OLLAMA_MAX_QUEUE'] = '100'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'
print("--> Starting Ollama...")
with open("ollama.log", "w") as log_file:
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)

print("--> Waiting for Ollama to be ready...")
for i in range(30):
    try:
        if requests.get("http://localhost:11434/").status_code == 200:
            print("--> Ollama is ready!")
            break
    except:
        time.sleep(1)
else:
    print("--> ERROR: Ollama did not start. Check ollama.log")

print("--> Pulling model (please wait)...")
!ollama pull smollm:135m
print("--> Model ready!")

In [ ]:
# Cell 3: Write the GPU Worker API to disk
api_code = '''
import os
import time
import asyncio
from typing import Optional
from contextlib import asynccontextmanager
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from ollama import Client

OLLAMA_HOST = "http://localhost:11434"
MODEL_NAME = "smollm:135m"
MAX_CONCURRENT_INFERENCE = 10

client = Client(host=OLLAMA_HOST)
inference_semaphore = asyncio.Semaphore(MAX_CONCURRENT_INFERENCE)

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Runs once at startup: set thread pool to match semaphore size
    loop = asyncio.get_event_loop()
    loop.set_default_executor(ThreadPoolExecutor(max_workers=MAX_CONCURRENT_INFERENCE))
    yield
    # Anything after yield runs at shutdown (nothing needed here)

app = FastAPI(title="GPU Worker API", version="1.0", lifespan=lifespan)

metrics_lock = asyncio.Lock()
metrics = {
    "active_requests": 0,
    "completed_requests": 0,
    "failed_requests": 0,
    "total_latency": 0.0,
    "total_queue_time": 0.0,
    "total_inference_time": 0.0,
}


class GenerateRequest(BaseModel):
    id: int
    query: str
    max_tokens: int = Field(default=64, ge=1, le=512)
    temperature: float = Field(default=0.2, ge=0.0, le=2.0)


class GenerateResponse(BaseModel):
    id: int
    model: str
    result: str
    queue_time: float
    inference_time: float
    total_latency: float


def run_ollama_inference(query: str, max_tokens: int, temperature: float) -> str:
    response = client.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": query}],
        options={"num_predict": max_tokens, "temperature": temperature},
    )
    if hasattr(response, "message"):
        return response.message.content
    return response["message"]["content"]


@app.get("/health")
async def health():
    return {
        "status": "ok",
        "model": MODEL_NAME,
        "max_concurrent_inference": MAX_CONCURRENT_INFERENCE,
    }


@app.get("/metrics")
async def get_metrics():
    async with metrics_lock:
        completed = metrics["completed_requests"]
        if completed > 0:
            average_latency = metrics["total_latency"] / completed
            average_queue_time = metrics["total_queue_time"] / completed
            average_inference_time = metrics["total_inference_time"] / completed
        else:
            average_latency = average_queue_time = average_inference_time = 0.0

        return {
            "model": MODEL_NAME,
            "max_concurrent_inference": MAX_CONCURRENT_INFERENCE,
            "active_requests": metrics["active_requests"],
            "completed_requests": metrics["completed_requests"],
            "failed_requests": metrics["failed_requests"],
            "average_latency": average_latency,
            "average_queue_time": average_queue_time,
            "average_inference_time": average_inference_time,
        }


@app.post("/generate", response_model=GenerateResponse)
async def generate(request: GenerateRequest):
    request_start = time.perf_counter()

    async with metrics_lock:
        metrics["active_requests"] += 1

    try:
        queue_start = time.perf_counter()
        async with inference_semaphore:
            queue_time = time.perf_counter() - queue_start

            inference_start = time.perf_counter()
            result = await asyncio.to_thread(
                run_ollama_inference,
                request.query,
                request.max_tokens,
                request.temperature,
            )
            inference_time = time.perf_counter() - inference_start

        total_latency = time.perf_counter() - request_start

        async with metrics_lock:
            metrics["completed_requests"] += 1
            metrics["total_latency"] += total_latency
            metrics["total_queue_time"] += queue_time
            metrics["total_inference_time"] += inference_time

        return GenerateResponse(
            id=request.id,
            model=MODEL_NAME,
            result=result,
            queue_time=queue_time,
            inference_time=inference_time,
            total_latency=total_latency,
        )

    except Exception as e:
        async with metrics_lock:
            metrics["failed_requests"] += 1
        raise HTTPException(status_code=500, detail=str(e))

    finally:
        async with metrics_lock:
            metrics["active_requests"] -= 1
'''

with open("gpu_worker_api.py", "w") as f:
    f.write(api_code)

print("--> gpu_worker_api.py written.")

In [ ]:
# Cell 4: Start the FastAPI server + expose via pyngrok tunnel
import subprocess
import time
import requests
from pyngrok import ngrok, conf
import os

# Kill any leftover uvicorn
subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
time.sleep(1)



os.environ["UVICORN_THREAD_POOL_SIZE"] = "10"
NGOK_AUTH_TOKEN = os.environ.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token("")

# Start the FastAPI worker in the background
print("--> Starting GPU Worker API on port 8000...")
with open("uvicorn.log", "w") as log_file:
    subprocess.Popen(
        ["uvicorn", "gpu_worker_api:app", "--host", "0.0.0.0", "--port", "8000"],
        stdout=log_file,
        stderr=log_file,
    )

# Wait for it to be up
print("--> Waiting for Worker API to be ready...")
for i in range(20):
    try:
        if requests.get("http://localhost:8000/health").status_code == 200:
            print("--> Worker API is ready!")
            break
    except:
        time.sleep(1)
else:
    print("--> ERROR: Worker API did not start. Check uvicorn.log")
    with open("uvicorn.log") as f:
        print(f.read())

# Open ngrok tunnel
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

# Clean the URL to get just the hostname (e.g., your-id.ngrok-free.app)
host = public_url.replace("https://", "").replace("http://", "")

print("\n" + "="*70)
print("CONNECTED SUCCESSFULLY (HTTP MODE)!")
print("="*70)
print(f"Public URL: {public_url}")
print(f"Hostname:   {host}")
print("\nUPDATE YOUR haproxy.cfg BACKEND WITH THE BLOCK BELOW:")
print("="*70)

# Show GPU info
print("\n--> GPU Status:")
!nvidia-smi

In [ ]:
response = requests.post("http://localhost:11434/api/generate", json={"model": "smollm:135m", "prompt": "hi", "stream": False})
print(response.text)
print("--> Model warmed up!")